# 03 — Shortest Paths (Dijkstra, Bellman-Ford, and Floyd-Warshall)

## Why This Matters for DSA
Finding the shortest path between nodes is one of the most widely applied problems in computer science. Whether it is calculating navigation routes (Google Maps), routing network packets across the internet (OSPF), or resolving optimal states in puzzles, shortest paths are at the center of the solution.

In Notebook 01, we saw that BFS can find the shortest path in an **unweighted** graph. However, real-world graphs have **weights** (e.g. latency, distance, toll cost). On weighted graphs, the BFS "closest in edges" metric no longer correlates with "closest in distance," and we must use more advanced routing strategies: **Dijkstra's**, **Bellman-Ford**, and **Floyd-Warshall**.

## Prerequisites
- `01_Representations_DFS_BFS` (Phase 04) — Graph representations (Adjacency list vs Matrix) and standard BFS queue traversals.
- `01_Heaps_and_Priority_Queues` (Phase 03) — Familiarity with C++ `std::priority_queue` operations.

## Learning Objectives
By the end of this notebook, you will be able to:
- Explain why Breadth-First Search (BFS) fails to find the shortest path on graphs with varying edge weights.
- Implement Dijkstra's Algorithm using a min-priority queue to achieve $O((V + E) \log V)$ complexity.
- Implement Bellman-Ford's Algorithm, state why it is needed when negative edge weights are present, and detect negative weight cycles.
- Implement Floyd-Warshall's Algorithm for all-pairs shortest paths using dynamic programming.
- Empirically analyze the performance gap between Dijkstra and Bellman-Ford on sparse graphs.

## Checklist Before Moving On
- [ ] I can explain why BFS fails on weighted graphs using a simple counterexample.
- [ ] I can write Dijkstra's algorithm from memory using C++ `std::priority_queue`.
- [ ] I can describe the "lazy deletion" (outdated key) optimization check in priority-queue-based Dijkstra.
- [ ] I can write Bellman-Ford's algorithm and explain why relaxing edges $V-1$ times is mathematically sufficient.
- [ ] I can write Floyd-Warshall's algorithm and explain the purpose of the three nested loops.
- [ ] I can identify negative cycles using Bellman-Ford.

## Table of Contents
1. Why BFS Fails on Weighted Graphs
2. Dijkstra's Algorithm — Single-Source Shortest Path (No Negative Weights)
3. Bellman-Ford Algorithm — Negative Edge Weights & Cycle Detection
4. Floyd-Warshall Algorithm — All-Pairs Shortest Path
5. Benchmarking: Dijkstra vs. Bellman-Ford
6. Checkpoint, Mini Project, and Practice Problems
7. LeetCode Practice Problems

## 1. Why BFS Fails on Weighted Graphs

### The Why
Breadth-First Search (BFS) is designed under a simple assumption: the distance to a node is directly proportional to its depth (number of edges from the source). If all edges have a cost of 1, BFS is perfectly optimal. 

However, if edge weights vary, a path with *fewer* edges can have a much *larger* total weight than a path with *more* edges.

### Core Mechanism
Consider this simple graph:
```
       (0) ---[w=10]--- (2)
        \             /
       [w=2]         [w=3]
          \         /
            - (1) -
```
If we start BFS from node `0`:
1. Node `0` pops. It visits its immediate neighbors: node `1` (dist = 2) and node `2` (dist = 10).
2. Node `2` is marked visited with distance `10`.
3. BFS completes.

However, the actual shortest path to node `2` is `0 -> 1 -> 2`, which has a total weight of $2 + 3 = 5$. Because BFS aggressively marks nodes as visited the first time they are reached, it locks in the path with fewer edges (`0 -> 2`), failing to discover the longer-edged, lower-weighted path.

To solve this, weighted shortest path algorithms utilize **relaxation**: repeatedly checking if we can improve our path to $v$ by traveling through $u$:
$$\text{dist}[v] = \min(\text{dist}[v], \text{dist}[u] + \text{weight}(u, v))$$

## 2. Dijkstra's Algorithm — Single-Source Shortest Path (No Negative Weights)

### The Why
Dijkstra's algorithm is a greedy algorithm that finds the shortest path from a single source vertex to all other vertices. It operates by maintaining a set of resolved vertices and greedily adding the closest unresolved vertex at each step. 

Dijkstra is highly efficient but comes with one critical restriction: **all edge weights must be non-negative**. If negative weights exist, Dijkstra's greedy assumption—that once a node is popped, its distance is finalized—breaks down.

### Core Mechanism
1. **Initialize:** Set `dist` of start node to `0`, and all other nodes to `INF`.
2. **Min-Priority Queue:** Use a min-heap to store pairs of `{distance, vertex}`. Push `{0, start}`.
3. **Iteration:** While the queue is not empty:
   - Extract the vertex $u$ with the minimum distance.
   - **Lazy Deletion Check:** If the popped distance $d > \text{dist}[u]$, skip it. This is an outdated queue entry.
   - For each neighbor $v$ of $u$ with edge weight $w$:
     - If $\text{dist}[u] + w < \text{dist}[v]$, relax the edge: $\text{dist}[v] = \text{dist}[u] + w$, and push `{dist[v], v}` to the queue.

### Common Pitfalls
- **Missing the Lazy Deletion Check:** In C++, `std::priority_queue` does not support updating keys dynamically. When we find a shorter path, we push a new `{new_dist, v}` pair, leaving the old `{old_dist, v}` pair in the queue. Without checking `if (d > dist[u]) continue;`, we will wastefully process neighbors of the same node multiple times, leading to severe slowdowns.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <queue>
#include <tuple>

using namespace std;

const int INF = 1e9; // represent infinity

class DijkstraGraph {
    int V;
    vector<vector<pair<int, int>>> adj; // pair of {neighbor, weight}

public:
    DijkstraGraph(int vertices) : V(vertices) {
        adj.resize(V);
    }

    void addEdge(int u, int v, int weight) {
        adj[u].push_back({v, weight});
    }

    vector<int> shortestPath(int start) {
        vector<int> dist(V, INF);
        // Min-priority queue: stores pair of {distance, vertex}
        priority_queue<pair<int, int>, vector<pair<int, int>>, greater<pair<int, int>>> pq;

        dist[start] = 0;
        pq.push({0, start});

        while (!pq.empty()) {
            auto [d, u] = pq.top();
            pq.pop();

            // Lazy deletion / Outdated pair check:
            if (d > dist[u]) continue;

            for (const auto& edge : adj[u]) {
                int v = edge.first;
                int weight = edge.second;

                // Relaxation Step
                if (dist[u] + weight < dist[v]) {
                    dist[v] = dist[u] + weight;
                    pq.push({dist[v], v});
                }
            }
        }
        return dist;
    }
};

int main() {
    cout << "--- Dijkstra's Algorithm Demo ---\n";
    DijkstraGraph g(5);
    g.addEdge(0, 1, 4);
    g.addEdge(0, 2, 2);
    g.addEdge(1, 2, 3);
    g.addEdge(1, 3, 2);
    g.addEdge(1, 4, 3);
    g.addEdge(2, 1, 1);
    g.addEdge(2, 3, 4);
    g.addEdge(2, 4, 5);
    g.addEdge(4, 3, 1);

    auto dist = g.shortestPath(0);
    for (int i = 0; i < 5; ++i) {
        cout << "Shortest distance from 0 to " << i << ": ";
        if (dist[i] == INF) cout << "INF\n";
        else cout << dist[i] << "\n";
    }

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. Bellman-Ford Algorithm — Negative Edge Weights & Cycle Detection

### The Why
When a graph contains negative edge weights, Dijkstra fails because it assumes that paths can only grow longer as we traverse edges. 

**Bellman-Ford** relaxes this greediness. It works by brute-force relaxation of *all* edges, repeating the process $V-1$ times. Furthermore, it is the primary tool for detecting **negative weight cycles**—loops whose sum of edge weights is negative. Traversing a negative cycle infinitely would result in a shortest path distance of $-\infty$.

### Core Mechanism
1. **Initialize:** Set `dist` of start node to `0`, and all other nodes to `INF`.
2. **Brute Force Relaxation:** Repeat $V-1$ times:
   - Loop through *every* edge $(u, v)$ with weight $w$.
   - If $\text{dist}[u]$ is not `INF` and $\text{dist}[u] + w < \text{dist}[v]$, relax the edge: $\text{dist}[v] = \text{dist}[u] + w$.
3. **Negative Cycle Check:** Run the edge relaxation one more time. If any distance $\text{dist}[v]$ decreases further, then that edge must belong to or be reachable from a negative cycle.

*Why $V-1$ times?* The longest possible simple path (path without loops) in a graph of $V$ vertices contains exactly $V-1$ edges. Thus, after $V-1$ iterations, the shortest paths are guaranteed to have stabilized, unless a negative loop is present.

### Common Pitfalls
- **INF Overflow:** If you initialize `INF` as `INT_MAX`, adding a negative edge weight in `dist[u] + weight` can cause an integer overflow. Always choose a safe `INF` value (e.g. `1e9`) or check `if (dist[u] != INF)` before relaxing.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>

using namespace std;

const int INF = 1e9;

struct Edge {
    int u, v, weight;
};

class BellmanFordGraph {
    int V;
    vector<Edge> edges;

public:
    BellmanFordGraph(int vertices) : V(vertices) {}

    void addEdge(int u, int v, int weight) {
        edges.push_back({u, v, weight});
    }

    // Returns {distances, hasNegativeCycle}
    pair<vector<int>, bool> shortestPath(int start) {
        vector<int> dist(V, INF);
        dist[start] = 0;

        // Relax all edges V-1 times
        for (int i = 0; i < V - 1; ++i) {
            for (const auto& edge : edges) {
                if (dist[edge.u] != INF && dist[edge.u] + edge.weight < dist[edge.v]) {
                    dist[edge.v] = dist[edge.u] + edge.weight;
                }
            }
        }

        // V-th iteration: Check for negative weight cycles
        bool hasNegativeCycle = false;
        for (const auto& edge : edges) {
            if (dist[edge.u] != INF && dist[edge.u] + edge.weight < dist[edge.v]) {
                hasNegativeCycle = true;
                break;
            }
        }

        return {dist, hasNegativeCycle};
    }
};

int main() {
    // 1. Graph without negative cycle
    cout << "--- Bellman-Ford without Negative Cycle ---\n";
    BellmanFordGraph g1(5);
    g1.addEdge(0, 1, -1);
    g1.addEdge(0, 2, 4);
    g1.addEdge(1, 2, 3);
    g1.addEdge(1, 3, 2);
    g1.addEdge(1, 4, 2);
    g1.addEdge(3, 2, 5);
    g1.addEdge(3, 1, 1);
    g1.addEdge(4, 3, -3);

    auto [dist1, cycle1] = g1.shortestPath(0);
    cout << "Has negative cycle? " << (cycle1 ? "Yes" : "No") << "\n";
    for (int i = 0; i < 5; ++i) {
        cout << "To " << i << ": " << dist1[i] << "\n";
    }

    // 2. Graph with negative cycle
    cout << "\n--- Bellman-Ford with Negative Cycle ---\n";
    BellmanFordGraph g2(4);
    g2.addEdge(0, 1, 1);
    g2.addEdge(1, 2, -1);
    g2.addEdge(2, 3, -1);
    g2.addEdge(3, 1, -1); // cycle: 1 -> 2 -> 3 -> 1 (sum: -1)

    auto [dist2, cycle2] = g2.shortestPath(0);
    cout << "Has negative cycle? " << (cycle2 ? "Yes" : "No") << "\n";

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. Floyd-Warshall Algorithm — All-Pairs Shortest Path

### The Why
Dijkstra and Bellman-Ford are **Single-Source** algorithms. If we need to find the shortest path between *all* pairs of vertices, we would have to run Dijkstra $V$ times (costing $O(V \cdot E \log V)$). For dense graphs, this can be complex to write.

**Floyd-Warshall** uses a clean Dynamic Programming matrix formulation to find all-pairs shortest paths in $O(V^3)$ time. It works by checking if paths can be shortened by inserting an intermediate node $k$ between $i$ and $j$.

### Core Mechanism
1. **Initialize Matrix:** Create a $V \times V$ matrix `dist` initialized to `INF`, with diagonal `dist[i][i] = 0`. For every direct edge $(u, v)$ with weight $w$, set `dist[u][v] = w`.
2. **DP Loop:** We iterate through all potential intermediate nodes $k$, and update the matrix:
```cpp
for (int k = 0; k < V; ++k) {
    for (int i = 0; i < V; ++i) {
        for (int j = 0; j < V; ++j) {
            dist[i][j] = min(dist[i][j], dist[i][k] + dist[k][j]);
        }
    }
}
```

### Common Pitfalls
- **Loop Ordering (Crucial Bug):** The intermediate node loop `k` **must** be the outermost loop. If you place `k` in the inner loops, the dynamic programming states will not resolve correctly, resulting in incorrect shortest path values.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>

using namespace std;

const int INF = 1e9;

class FloydWarshall {
    int V;
    vector<vector<int>> dist;

public:
    FloydWarshall(int vertices) : V(vertices) {
        dist.assign(V, vector<int>(V, INF));
        for (int i = 0; i < V; ++i) {
            dist[i][i] = 0; // Self distance is 0
        }
    }

    void addEdge(int u, int v, int weight) {
        dist[u][v] = weight;
    }

    void run() {
        for (int k = 0; k < V; ++k) {
            for (int i = 0; i < V; ++i) {
                for (int j = 0; j < V; ++j) {
                    if (dist[i][k] != INF && dist[k][j] != INF) {
                        dist[i][j] = min(dist[i][j], dist[i][k] + dist[k][j]);
                    }
                }
            }
        }
    }

    void print() const {
        cout << "All-Pairs Shortest Path Matrix:\n";
        for (int i = 0; i < V; ++i) {
            for (int j = 0; j < V; ++j) {
                if (dist[i][j] == INF) cout << "INF ";
                else cout << dist[i][j] << " ";
            }
            cout << "\n";
        }
    }
};

int main() {
    cout << "--- Floyd-Warshall Demo ---\n";
    FloydWarshall fw(4);
    fw.addEdge(0, 3, 10);
    fw.addEdge(0, 1, 5);
    fw.addEdge(1, 2, 3);
    fw.addEdge(2, 3, 1);

    fw.run();
    fw.print();

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. Benchmarking: Dijkstra vs. Bellman-Ford

### The Why
To show the empirical speed difference between Dijkstra's $O(E \log V)$ and Bellman-Ford's $O(V \cdot E)$ complexity, we run a micro-benchmark on a sparse graph of $V = 1000, E = 5000$.

Let's compile and run the benchmark.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <queue>
#include <chrono>
#include <random>

using namespace std;

const int INF = 1e9;

struct Timer {
    string name;
    chrono::high_resolution_clock::time_point start;
    Timer(const string& n) : name(n), start(chrono::high_resolution_clock::now()) {}
    ~Timer() {
        auto end = chrono::high_resolution_clock::now();
        auto diff = chrono::duration_cast<chrono::microseconds>(end - start).count();
        cout << name << ": " << diff << " microseconds\n";
    }
};

struct Edge {
    int u, v, weight;
};

int main() {
    const int V = 1000;
    const int E = 5000;

    mt19937 rng(42);
    uniform_int_distribution<int> distV(0, V - 1);
    uniform_int_distribution<int> distW(1, 100);

    vector<Edge> edges;
    vector<vector<pair<int, int>>> adj(V);

    for (int i = 0; i < E; ++i) {
        int u = distV(rng);
        int v = distV(rng);
        int w = distW(rng);
        if (u != v) {
            edges.push_back({u, v, w});
            adj[u].push_back({v, w});
        }
    }

    // Benchmark Dijkstra
    {
        Timer t("Dijkstra's Algorithm");
        vector<int> dist(V, INF);
        priority_queue<pair<int, int>, vector<pair<int, int>>, greater<pair<int, int>>> pq;
        dist[0] = 0;
        pq.push({0, 0});
        while (!pq.empty()) {
            auto [d, u] = pq.top();
            pq.pop();
            if (d > dist[u]) continue;
            for (const auto& edge : adj[u]) {
                int v = edge.first;
                int w = edge.second;
                if (dist[u] + w < dist[v]) {
                    dist[v] = dist[u] + w;
                    pq.push({dist[v], v});
                }
            }
        }
        cout << "Dijkstra final distance to V-1: " << dist[V-1] << "\n";
    }

    // Benchmark Bellman-Ford
    {
        Timer t("Bellman-Ford Algorithm");
        vector<int> dist(V, INF);
        dist[0] = 0;
        for (int i = 0; i < V - 1; ++i) {
            for (const auto& edge : edges) {
                if (dist[edge.u] != INF && dist[edge.u] + edge.weight < dist[edge.v]) {
                    dist[edge.v] = dist[edge.u] + edge.weight;
                }
            }
        }
        cout << "Bellman-Ford final distance to V-1: " << dist[V-1] << "\n";
    }

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. Checkpoint, Cheat Sheet, and Answer Key

### Complexity & Limits Summary

| Algorithm | Complexity | Source Type | Weights | Negative Cycle Detection |
|---|---|---|---|---|
| **Dijkstra** | $O((V+E) \log V)$ | Single-Source | Non-Negative | No (breaks completely) |
| **Bellman-Ford** | $O(V \cdot E)$ | Single-Source | Any weight | Yes (V-th iteration) |
| **Floyd-Warshall** | $O(V^3)$ | All-Pairs | Any weight | Yes (check `dist[i][i] < 0`) |

### Checkpoint Questions
1. Why does Dijkstra's algorithm fail to find correct shortest paths if negative weight edges are present?
2. Why is the lazy deletion step (`d > dist[u]`) important in heap-based Dijkstra?
3. How can we check for negative cycles using Floyd-Warshall?
4. Why does Bellman-Ford require exactly $V-1$ iterations?
5. Under what conditions is running Dijkstra $V$ times faster than running Floyd-Warshall once for All-Pairs Shortest Paths?
6. Can a shortest path change if we add a positive constant $C$ to every edge weight in the graph?

### Answer Key
1. Dijkstra is greedy: once a node is popped from the priority queue, Dijkstra assumes its shortest path is locked. If a negative weight edge is encountered later, it can reduce the cost of a path to an already-locked node, which Dijkstra won't update.
2. In C++, we cannot update keys in `std::priority_queue`. When a shorter path to $u$ is found, we push it to the queue. The old, longer path remains. Lazy deletion ignores these duplicate, outdated entries, saving $O(E \log V)$ redundant traversals.
3. If `dist[i][i] < 0` for any vertex $i$ after running the algorithm, a negative cycle containing vertex $i$ exists.
4. The longest path without cycles in a graph with $V$ vertices has at most $V-1$ edges. Since each relaxation iteration propagates distance information by 1 edge, $V-1$ runs are sufficient to calculate the final shortest path values.
5. On sparse graphs where $E \ll V^2$, running Dijkstra $V$ times costs $O(V \cdot E \log V)$. Floyd-Warshall costs $O(V^3)$. If $E \approx V$, running Dijkstra $V$ times is $O(V^2 \log V)$, which is significantly faster than $O(V^3)$.
6. Yes. Paths with *more* edges will have their total weight increased by more than paths with *fewer* edges. Thus, a path that had 3 edges and sum 5, and another path with 1 edge and sum 6, will change to sums $5+3C$ and $6+C$ respectively. If $C > 0.5$, the 1-edge path becomes the new shortest path.

### Mini Project: Network Router Latency Simulator
Build a simulator where routers represent nodes and latencies represent edge weights.
Write functions to:
1. Run Dijkstra's algorithm to calculate optimal routing paths and latencies.
2. Reconstruct the routing sequence of routers.
3. Introduce a priority shortcut (negative weight) and run Bellman-Ford to check for routing cycles.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <queue>
#include <string>
#include <unordered_map>
#include <algorithm>

using namespace std;

const int INF = 1e9;

struct Link {
    int u, v, weight;
};

class RouterNetwork {
    int V;
    vector<vector<pair<int, int>>> adj; // pair of {neighbor, latency}
    vector<Link> links;
    unordered_map<int, string> idToName;
    unordered_map<string, int> nameToId;

public:
    RouterNetwork(const vector<string>& routers) {
        V = routers.size();
        adj.resize(V);
        for (int i = 0; i < V; ++i) {
            idToName[i] = routers[i];
            nameToId[routers[i]] = i;
        }
    }

    void addLink(const string& from, const string& to, int latency) {
        int u = nameToId[from];
        int v = nameToId[to];
        adj[u].push_back({v, latency});
        links.push_back({u, v, latency});
    }

    // 1. Dijkstra for fastest routing path
    pair<vector<int>, vector<int>> getFastestRoutes(const string& start) {
        int startId = nameToId[start];
        vector<int> dist(V, INF);
        vector<int> parent(V, -1);
        priority_queue<pair<int, int>, vector<pair<int, int>>, greater<pair<int, int>>> pq;

        dist[startId] = 0;
        pq.push({0, startId});

        while (!pq.empty()) {
            auto [d, u] = pq.top();
            pq.pop();

            if (d > dist[u]) continue;

            for (const auto& edge : adj[u]) {
                int v = edge.first;
                int w = edge.second;
                if (dist[u] + w < dist[v]) {
                    dist[v] = dist[u] + w;
                    parent[v] = u;
                    pq.push({dist[v], v});
                }
            }
        }
        return {dist, parent};
    }

    // 2. Bellman-Ford for negative weight check and routing loop detection
    pair<vector<int>, bool> checkNetworkLoops(const string& start) {
        int startId = nameToId[start];
        vector<int> dist(V, INF);
        dist[startId] = 0;

        for (int i = 0; i < V - 1; ++i) {
            for (const auto& l : links) {
                if (dist[l.u] != INF && dist[l.u] + l.weight < dist[l.v]) {
                    dist[l.v] = dist[l.u] + l.weight;
                }
            }
        }

        bool hasNegativeCycle = false;
        for (const auto& l : links) {
            if (dist[l.u] != INF && dist[l.u] + l.weight < dist[l.v]) {
                hasNegativeCycle = true;
                break;
            }
        }
        return {dist, hasNegativeCycle};
    }

    vector<string> reconstructPath(const string& start, const string& dest, const vector<int>& parent) {
        vector<string> path;
        int curr = nameToId[dest];
        int startId = nameToId[start];
        if (parent[curr] == -1 && curr != startId) return path;

        while (curr != -1) {
            path.push_back(idToName[curr]);
            curr = parent[curr];
        }
        reverse(path.begin(), path.end());
        return path;
    }
};

int main() {
    vector<string> routers = {"A", "B", "C", "D", "E"};
    RouterNetwork net(routers);

    // Initial normal latencies
    net.addLink("A", "B", 4);
    net.addLink("A", "C", 2);
    net.addLink("B", "D", 5);
    net.addLink("C", "B", 1);
    net.addLink("C", "D", 8);
    net.addLink("C", "E", 10);
    net.addLink("D", "E", 2);

    cout << "--- Router Network Simulator (Normal Dijkstra Routing) ---\n";
    auto [dist, parent] = net.getFastestRoutes("A");
    auto path = net.reconstructPath("A", "E", parent);
    cout << "Fastest path A -> E: ";
    for (const auto& name : path) cout << name << " ";
    cout << "\nTotal latency: " << dist[net.nameToId["E"]] << " ms\n"; // Expected: A -> C -> B -> D -> E (Total: 10 ms)

    cout << "\n--- Introducing Priority Channel Bypass and Back-Link (Bellman-Ford Checking) ---\n";
    RouterNetwork net2(routers);
    net2.addLink("A", "B", 4);
    net2.addLink("A", "C", 2);
    net2.addLink("B", "D", 5);
    net2.addLink("C", "B", 1);
    net2.addLink("C", "D", 8);
    net2.addLink("C", "E", 10);
    net2.addLink("D", "E", 2);

    // Add a negative channel: D -> B with -10 ms latency, creating a loop B -> D -> B of weight 5 - 10 = -5 ms
    net2.addLink("D", "B", -10);

    auto [dist2, hasLoop] = net2.checkNetworkLoops("A");
    cout << "Routing Loop (Negative Cycle) Detected? " << (hasLoop ? "Yes" : "No") << "\n"; // Expected: Yes

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. LeetCode Practice Problems

| # | Problem | Difficulty | Topic | Hint |
|---|---|---|---|---|
| 743 | Network Delay Time | Medium | Dijkstra | Find the time it takes for all nodes to receive a signal from a source. Standard Dijkstra's algorithm. |
| 1514 | Path with Maximum Probability | Medium | Dijkstra Variant | Find path with max probability. Multiply probabilities instead of adding weights. Since probabilities $\le 1$, greedy choices still work. Use a max-heap. |
| 787 | Cheapest Flights Within K Stops | Medium | Bellman-Ford Variant | Find shortest path with at most $k$ edges. Run Bellman-Ford exactly $k+1$ times, making sure to use a copy of the distance array to prevent edge relaxation propagation within the same loop iteration. |
| 1334 | Find the City With the Smallest Number of Neighbors at a Threshold Distance | Medium | Floyd-Warshall | Run Floyd-Warshall to get APSP matrix. Count cities reachable within threshold distance, and return the city with the minimum count. |
| 1631 | Path With Minimum Effort | Medium | Dijkstra Variant | Find path where the maximum absolute difference between consecutive cells is minimized. Relax edges using `min(dist[v], max(dist[u], height_diff))`. |
| 2093 | Minimum Cost to Reach City With Discounts | Medium | Dijkstra State | Track distance as a 2D state: `dist[u][discounts_used]`. Relax transitions using a discount or no discount. |
| 1976 | Number of Ways to Arrive at Destination | Medium | Dijkstra DP | Keep an auxiliary `ways[]` array. During Dijkstra relaxation: if a shorter path is found, set `ways[v] = ways[u]`. If an equal path is found, add `ways[v] += ways[u]`. |

### Self-Check Before Moving to `04_Minimum_Spanning_Trees`
- Can you explain why Dijkstra's algorithm fails with negative weights?
- Do you know how to detect negative cycles using both Bellman-Ford and Floyd-Warshall?
- Can you write the outer loop variable for Floyd-Warshall?

If you feel confident, progress to **04_Minimum_Spanning_Trees.ipynb**, where we explore spanning tree algorithms (Kruskal's and Prim's) and their relation to dynamic components.